# UniversityScope · Global University Intelligence
**Blessing Sekeramavi | Data science & international outreach**

An interactive dashboard with a real 3D country globe, flat world map, geographic coverage, performance exploration, institution comparisons and CSV exports.

**Start:** In Colab choose **Runtime → Run all**. The dashboard appears inside the final cell. Dependencies and external map libraries need internet access. In GitHub, JavaScript dashboards are disabled: use the **Open in Colab** link in the README.

**Location accuracy:** every recognized institution contributes to its country aggregate. Country polygons are not campus coordinates. No synthetic university locations or invented scores are used. Rankings are screening evidence, not a measure of partnership readiness.


In [ ]:
%pip -q install "pandas>=2.2,<3" "numpy>=1.26,<3" "requests>=2.31,<3" "beautifulsoup4>=4.12,<5" "pycountry>=24.6" "plotly>=5.20,<7" "scikit-learn>=1.4,<2"


## 1 · Load a saved CSV or retrieve the source
Upload `times_higher_education_rankings.csv` through the Colab Files sidebar to reuse your existing snapshot. Otherwise the next cell attempts the THE latest ranking page. A changed source layout produces an actionable error. The ranking edition remains unverified until checked against the publisher.


In [1]:
"""UniversityScope: source loading and conservative numeric parsing."""
from pathlib import Path
import json
import re
import pandas as pd

SOURCE = 'https://www.timeshighereducation.com/world-university-rankings/latest/world-ranking'

def bounds(value):
    if pd.isna(value):
        return None, None
    text = str(value).strip().lstrip('=')
    if re.fullmatch(r'\d+(?:\.\d+)?', text):
        return float(text), float(text)
    match = re.fullmatch(r'(\d+(?:\.\d+)?)\s*[-–—]\s*(\d+(?:\.\d+)?)', text)
    if match:
        low, high = map(float, match.groups())
        return (low, high) if low <= high else (None, None)
    if re.fullmatch(r'\d+\s*\+', text):
        return float(text.rstrip('+').strip()), None
    return None, None

def clean(raw):
    required = {'name', 'location', 'rank'}
    if not required.issubset(raw.columns):
        raise ValueError(f'Missing columns: {required - set(raw.columns)}')
    df = raw.copy().reset_index(drop=True)
    df['name'] = df['name'].fillna('Unnamed institution').astype(str).str.strip()
    df['location'] = df['location'].fillna('Unknown').astype(str).str.strip()
    df['rank_original'] = df['rank'].fillna('Unknown').astype(str)
    pairs = df['rank'].apply(bounds)
    df['rank_lower_bound'] = [p[0] for p in pairs]
    df['rank_upper_bound'] = [p[1] for p in pairs]
    disabled = df.get('disabled', pd.Series(False, index=df.index)).astype(str).str.lower().isin(['true', '1'])
    df['academic_status'] = 'Unknown/unranked'
    df.loc[df['rank_lower_bound'].notna(), 'academic_status'] = 'Ranked'
    df.loc[df['rank_original'].str.contains('reporter', case=False), 'academic_status'] = 'Reporter'
    df.loc[disabled, 'academic_status'] = 'Disabled'
    for key in ['overall', 'teaching', 'research', 'citations', 'industry_income', 'international_outlook']:
        parsed = df.get('scores_' + key, pd.Series(None, index=df.index, dtype=object)).apply(bounds)
        for pos, label in enumerate(['lower', 'upper']):
            df[f'score_{key}_{label}'] = pd.to_numeric(pd.Series([p[pos] for p in parsed]), errors='coerce')
            df.loc[~df[f'score_{key}_{label}'].between(0, 100), f'score_{key}_{label}'] = float('nan')
    return df

def load_data(path='times_higher_education_rankings.csv'):
    path = Path(path)
    if path.exists():
        return pd.read_csv(path), f'Local file: {path.name}; edition not verified'
    import requests
    from bs4 import BeautifulSoup
    response = requests.get(SOURCE, headers={'User-Agent': 'UniversityScope portfolio research'}, timeout=45)
    response.raise_for_status()
    script = BeautifulSoup(response.text, 'html.parser').find('script', id='__NEXT_DATA__')
    if not script:
        raise RuntimeError('Source layout changed. Upload your previously saved rankings CSV and rerun.')
    payload = json.loads(script.string)
    try:
        records = payload['props']['pageProps']['page']['rankingsTableConfig']['rankingsData']['data']
    except KeyError as error:
        raise RuntimeError('Source data path changed. Upload the original CSV and rerun.') from error
    raw = pd.DataFrame(records)
    raw.to_csv(path, index=False)
    return raw, 'THE latest URL; ranking edition not verified'

raw, provenance = load_data()
df = clean(raw)
df.to_csv('times_higher_education_cleaned.csv', index=False)
print(f"Loaded {len(df):,} institutions · {df.location.nunique():,} locations")
display(df.academic_status.value_counts().rename_axis('Status').to_frame('Institutions'))


Loaded 3,146 institutions · 131 locations


,Institutions
Status,
Ranked,2224
Reporter,821
Disabled,101


## 2 · Quality checks and explainable clustering
Rank bands retain their bounds. Open-ended bands retain the lower bound and an unknown upper bound. Missing performance values remain missing. Clustering uses only complete component-score records and neutral labels. It describes score profiles; it does not establish institutional missions.


In [2]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

features = ['score_' + m + '_lower' for m in ['teaching','research','citations','industry_income','international_outlook']]
ranked = df.loc[df.academic_status.eq('Ranked')].copy()
missing = ranked[features].isna().mean().mul(100).round(1)
display(missing.rename('Missing % among ranked').to_frame())
complete = ranked.dropna(subset=features).copy()
print(f'Clustering eligible: {len(complete):,} / {len(ranked):,} ranked institutions (complete cases).')
if len(complete) >= 10:
    X = StandardScaler().fit_transform(complete[features])
    models, evaluations = {}, []
    for k in range(2, min(6, len(complete))):
        model = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
        if 1 < len(set(model.labels_)) < len(X):
            score = silhouette_score(X, model.labels_, sample_size=min(2000, len(X)), random_state=42)
            models[k] = model
            evaluations.append({'k': k, 'silhouette': score})
    if evaluations:
        evaluation = pd.DataFrame(evaluations).sort_values('silhouette', ascending=False)
        display(evaluation)
        best_k = int(evaluation.iloc[0]['k'])
        complete['cluster'] = models[best_k].labels_
        display(complete.groupby('cluster')[features].mean().round(1))
        complete[['name','location','rank_original','cluster'] + features].to_csv('universityscope_clusters.csv', index=False)
        print(f'Selected k={best_k} by silhouette; sensitivity and external validity remain untested.')
    else:
        print('Not enough distinct profiles for meaningful clustering.')
else:
    print('Too few complete profiles for clustering. The dashboard remains available.')


Clustering eligible: 2,224 / 2,224 ranked institutions (complete cases).
Selected k=2 by silhouette; sensitivity and external validity remain untested.


,Missing % among ranked
score_teaching_lower,0.0
score_research_lower,0.0
score_citations_lower,0.0
score_industry_income_lower,0.0
score_international_outlook_lower,0.0


,k,silhouette
0,2,0.434981
1,3,0.345643
3,5,0.303280
2,4,0.291045


,score_teaching_lower,score_research_lower,score_citations_lower,score_industry_income_lower,score_international_outlook_lower
cluster,,,,,
0,22.8,16.6,43.2,33.7,41.2
1,41.7,41.7,75.6,77.5,68.6


## 3 · Launch the interactive dashboard
The next cell creates `universityscope_dashboard.html` and **displays it here**. The complete dashboard runs inside its own frame: scroll within the frame to reach comparisons and the institution table. It uses country counts for mapping and lower score bounds for comparison charts.

If an external library cannot load, an error is displayed. You can also download the generated HTML from the Files sidebar and open it in a browser with internet access. GitHub notebook previews cannot execute its JavaScript.


In [ ]:
from pathlib import Path
Path('dashboard.html').write_text('<!doctype html>\n<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>UniversityScope | Global intelligence</title>\n<style>\n:root{color-scheme:dark;font-family:Inter,system-ui,sans-serif;background:#080d1a;color:#e8eefb}*{box-sizing:border-box}body{margin:0}header{padding:27px 30px 16px;border-bottom:1px solid #26314b;display:flex;justify-content:space-between;gap:20px;align-items:center}h1{font-size:28px;letter-spacing:-1px;margin:3px 0}h2{font-size:17px;margin:0 0 13px}p{color:#a5b5d0;font-size:13px;line-height:1.6;margin:6px 0}.eyebrow{color:#57d6cb;font-size:11px;letter-spacing:3px}.tag{border:1px solid #33425f;border-radius:20px;padding:8px 13px;color:#9bded8;font-size:12px}main{padding:20px 30px}.toolbar{display:flex;gap:12px;flex-wrap:wrap;align-items:end;margin:16px 0}label{font-size:12px;color:#a5b5d0;display:flex;flex-direction:column;gap:6px}input,select,button{font:inherit;padding:10px 12px;border:1px solid #33425f;border-radius:8px;background:#111c31;color:#e8eefb}button{cursor:pointer}button.active{background:#216e77;border-color:#63ddd1}button:focus-visible,input:focus-visible,select:focus-visible{outline:2px solid #63ddd1}.stats{display:grid;grid-template-columns:repeat(4,1fr);gap:14px}.stat{border-radius:12px;background:#111a2d;padding:17px}.number{font-size:28px;font-weight:650;margin-bottom:4px}.stat span{font-size:12px;color:#a5b5d0}.map-shell{background:radial-gradient(ellipse at center,#132840,#080d1a 70%);border:1px solid #26314b;border-radius:15px;overflow:hidden}.map-head{padding:18px;display:flex;justify-content:space-between;gap:10px;align-items:center}#map{height:470px;position:relative}#globe,#flat{position:absolute;inset:0}#flat{display:none}.map-note{padding:12px 18px;border-top:1px solid #26314b;color:#a5b5d0;font-size:12px}.charts{display:grid;grid-template-columns:1fr 1fr;gap:16px;margin:17px 0}.panel{background:#10192c;border:1px solid #26314b;border-radius:13px;padding:18px}.chart{height:300px}.table-wrap{max-height:380px;overflow:auto}table{border-collapse:collapse;width:100%;font-size:12px}th{text-align:left;color:#9fb0ce;background:#14213a;position:sticky;top:0}td,th{padding:11px 10px;border-bottom:1px solid #26314b}td:first-child{font-weight:550}#status{color:#ffcc84;min-height:18px;font-size:12px}.compare-controls{display:flex;gap:10px;flex-wrap:wrap}footer{padding:18px 0;color:#8193b0;font-size:12px}@media(max-width:760px){header,main{padding:18px}.stats{grid-template-columns:1fr 1fr}.charts{grid-template-columns:1fr}h1{font-size:24px}.tag{display:none}.toolbar label{flex:1;min-width:140px}.map-head{align-items:start;flex-direction:column}#map{height:390px}}\n</style>\n<script src="https://cdn.jsdelivr.net/npm/plotly.js-dist-min@2.35.2/plotly.min.js"></script>\n<script src="https://cdn.jsdelivr.net/npm/globe.gl@2.33.2/dist/globe.gl.min.js"></script>\n<script src="https://cdn.jsdelivr.net/npm/topojson-client@3.1.0/dist/topojson-client.min.js"></script>\n</head><body>\n<header><div><div class="eyebrow">HIGHER EDUCATION · DATA INTELLIGENCE</div><h1>UniversityScope</h1><p>Explore the world of universities. Find patterns. Compare performance.</p></div><div class="tag">Research portfolio / Blessing Sekeramavi</div></header>\n<main>\n<div class="stats"><div class="stat"><div class="number" id="total">—</div><span>Institutions in selection</span></div><div class="stat"><div class="number" id="ranked">—</div><span>Ranked institutions</span></div><div class="stat"><div class="number" id="countries">—</div><span>Countries / territories</span></div><div class="stat"><div class="number" id="coverage">—</div><span>With selected metric</span></div></div>\n<div class="toolbar"><label>Search institution<input id="search" placeholder="Oxford, Constructor, Tokyo…" type="search"></label><label>Country / territory<select id="country"><option value="">All locations</option></select></label><label>Institution status<select id="cohort"><option value="">All records</option><option>Ranked</option><option>Reporter</option><option>Disabled</option><option>Unknown/unranked</option></select></label><label>Performance dimension<select id="metric"><option value="overall">Overall</option><option value="teaching">Teaching</option><option value="research">Research</option><option value="citations">Citations</option><option value="international_outlook">International outlook</option><option value="industry_income">Industry income</option></select></label><button id="reset">Reset</button><button id="export">Export selection ↓</button></div>\n<div id="status" role="status" aria-live="polite"></div>\n<section class="map-shell"><div class="map-head"><div><h2>Global university footprint</h2><p id="mapdetail">Country totals • hover for coverage • click to filter</p></div><div><button id="globeBtn" class="active" aria-pressed="true">3D globe</button> <button id="flatBtn" aria-pressed="false">2D map</button></div></div><div id="map"><div id="globe"></div><div id="flat"></div></div><div class="map-note" id="mapnote">Country-level aggregates represent institutions with a recognized location. They are not university campus coordinates. Drag the globe to explore; scroll to zoom.</div></section>\n<div class="charts"><section class="panel"><h2>Where institutions are represented</h2><div id="bars" class="chart"></div></section><section class="panel"><h2>Teaching × research</h2><div id="scatter" class="chart"></div></section></div>\n<section class="panel"><h2>Compare institutions</h2><p>Choose up to four. Values show lower score bounds; missing scores remain missing.</p><div id="compareControls" class="compare-controls"></div><div id="compare" class="chart"></div></section>\n<section class="panel" style="margin-top:17px"><h2>Institution explorer</h2><p id="tableNote"></p><div class="table-wrap"><table><thead><tr><th>Institution</th><th>Location</th><th>Rank / band</th><th>Status</th><th>Score range</th></tr></thead><tbody id="rows"></tbody></table></div></section>\n<footer id="source"></footer>\n</main>\n<script>\nconst payload=__PAYLOAD__,data=payload.rows,countryCodes=payload.countries;\nconst $=id=>document.getElementById(id),esc=v=>String(v??\'\').replace(/[&<>"\']/g,c=>({\'&\':\'&amp;\',\'<\':\'&lt;\',\'>\':\'&gt;\',\'"\':\'&quot;\',"\'":\'&#39;\'}[c]));\nconst finite=v=>typeof v===\'number\'&&Number.isFinite(v),fmt=v=>finite(v)?v.toFixed(1):\'—\';\nlet selected=data,features=[],globe=null,currentView=\'globe\',comparison=[];\nconst layout={paper_bgcolor:\'transparent\',plot_bgcolor:\'transparent\',font:{color:\'#b6c5df\',family:\'system-ui\',size:11},margin:{l:60,r:15,t:15,b:45},xaxis:{gridcolor:\'#23314b\',zeroline:false},yaxis:{gridcolor:\'#23314b\',zeroline:false},colorway:[\'#5ee0ce\',\'#9c91ff\',\'#ffbf7b\',\'#6bb8ff\']};\nconst config={responsive:true,displayModeBar:false};\nfunction plot(id,traces,extra={}){if(window.Plotly)Plotly.react(id,traces,{...layout,...extra},config)}\nfunction aggregate(){const out={};selected.forEach(r=>{const code=countryCodes[r.location];if(!code)return;const key=String(Number(code.numeric));if(!out[key])out[key]={count:0,name:r.location,iso:code.iso,values:[]};out[key].count++;const value=r[\'score_\'+$(\'metric\').value+\'_lower\'];if(finite(value))out[key].values.push(value)});return out}\nfunction color(n,max){return n?`rgba(71,${Math.round(115+105*Math.sqrt(n/max))},${Math.round(158+48*Math.sqrt(n/max))},0.92)`:\'#182741\'}\nfunction updateMap(){const agg=aggregate(),max=Math.max(1,...Object.values(agg).map(x=>x.count));if(globe){globe.polygonsData(features).polygonCapColor(f=>color(agg[String(Number(f.id))]?.count||0,max)).polygonAltitude(f=>agg[String(Number(f.id))]?0.015+0.12*Math.sqrt(agg[String(Number(f.id))].count/max):0.005).polygonLabel(f=>{const a=agg[String(Number(f.id))];return `<div style="padding:10px;background:#111c31;color:#e8eefb">${esc(a?.name||f.properties.name)}<br>${a?.count||0} institutions in selection<br>Median ${esc($(\'metric\').selectedOptions[0].text)} lower bound: ${a?.values.length?fmt(median(a.values)):\'unavailable\'}</div>`}).onPolygonClick(f=>{const a=agg[String(Number(f.id))];if(a){$(\'country\').value=a.name;render()}})}\nconst entries=Object.values(agg);plot(\'flat\',[{type:\'choropleth\',locationmode:\'ISO-3\',locations:entries.map(x=>x.iso),z:entries.map(x=>x.count),text:entries.map(x=>x.name),colorscale:[[0,\'#183653\'],[1,\'#5ee0ce\']],colorbar:{title:\'Institutions\',thickness:10},hovertemplate:\'%{text}<br>%{z} institutions<extra></extra>\',marker:{line:{color:\'#344761\',width:0.4}}}],{geo:{projection:{type:\'natural earth\'},bgcolor:\'transparent\',showframe:false,showcoastlines:false,showland:true,landcolor:\'#16243c\',countrycolor:\'#344761\',showcountries:true},margin:{l:0,r:0,t:0,b:0}});\nif(window.Plotly&&$(\'flat\').on&&!$(\'flat\').dataset.clickBound){$(\'flat\').on(\'plotly_click\',e=>{const name=e.points?.[0]?.text;if(name){$(\'country\').value=name;render()}});$(\'flat\').dataset.clickBound=\'1\'}\nconst mapped=Object.values(agg).reduce((a,b)=>a+b.count,0);$(\'mapnote\').textContent=`${mapped.toLocaleString()} of ${selected.length.toLocaleString()} selected institutions matched to ISO country codes. ${selected.length-mapped} have unresolved country codes. Small territories may not appear in the boundary geometry. These are not campus locations. Heights and colours show counts.`}\nfunction median(values){const s=[...values].sort((a,b)=>a-b);return s.length%2?s[(s.length-1)/2]:(s[s.length/2-1]+s[s.length/2])/2}\nfunction render(){const term=$(\'search\').value.toLocaleLowerCase(),country=$(\'country\').value,status=$(\'cohort\').value,key=\'score_\'+$(\'metric\').value+\'_lower\';selected=data.filter(r=>(!term||r.name.toLocaleLowerCase().includes(term))&&(!country||r.location===country)&&(!status||r.academic_status===status));$(\'total\').textContent=selected.length.toLocaleString();$(\'ranked\').textContent=selected.filter(r=>r.academic_status===\'Ranked\').length.toLocaleString();$(\'countries\').textContent=new Set(selected.map(r=>r.location)).size;$(\'coverage\').textContent=selected.filter(r=>finite(r[key])).length.toLocaleString();\nconst counts={};selected.forEach(r=>counts[r.location]=(counts[r.location]||0)+1);const top=Object.entries(counts).sort((a,b)=>b[1]-a[1]).slice(0,12).reverse();plot(\'bars\',[{type:\'bar\',orientation:\'h\',x:top.map(x=>x[1]),y:top.map(x=>x[0]),marker:{color:\'#5ed8cb\'},hovertemplate:\'%{y}: %{x} institutions<extra></extra>\'}],{margin:{l:125,r:15,t:5,b:40},xaxis:{...layout.xaxis,title:\'Institutions\'},yaxis:{...layout.yaxis,automargin:true}});\nconst valid=selected.filter(r=>r.academic_status===\'Ranked\'&&finite(r.score_teaching_lower)&&finite(r.score_research_lower));plot(\'scatter\',[{type:\'scattergl\',mode:\'markers\',x:valid.map(r=>r.score_teaching_lower),y:valid.map(r=>r.score_research_lower),text:valid.map(r=>esc(r.name)+\'<br>\'+esc(r.location)+\' · \'+esc(r.rank_original)),marker:{size:6,opacity:0.65,color:valid.map(r=>finite(r[key])?r[key]:null),colorscale:\'Viridis\',showscale:false},hovertemplate:\'%{text}<br>Teaching: %{x}<br>Research: %{y}<extra></extra>\'}],{xaxis:{...layout.xaxis,title:\'Teaching · lower score bound\',range:[0,100]},yaxis:{...layout.yaxis,title:\'Research · lower score bound\',range:[0,100]},annotations:valid.length?[]:[{text:\'No complete teaching / research pairs\',showarrow:false,x:0.5,y:0.5,xref:\'paper\',yref:\'paper\'}]});\n$(\'rows\').innerHTML=selected.slice(0,250).map(r=>{const lo=r[key],hi=r[key.replace(\'_lower\',\'_upper\')];return `<tr><td>${esc(r.name)}</td><td>${esc(r.location)}</td><td>${esc(r.rank_original)}</td><td>${esc(r.academic_status)}</td><td>${finite(lo)?fmt(lo)+(finite(hi)&&hi!==lo?\'–\'+fmt(hi):\'\'):\'—\'}</td></tr>`}).join(\'\');$(\'tableNote\').textContent=`${selected.length.toLocaleString()} matches · showing first ${Math.min(selected.length,250)} · export includes all matches · ${$(\'metric\').selectedOptions[0].text} scores.`;updateMap();compare()}\nfunction compare(){const metrics=[\'teaching\',\'research\',\'citations\',\'international_outlook\',\'industry_income\'];const traces=comparison.filter(x=>x!==\'\').map(idx=>{const r=data[Number(idx)];return{type:\'bar\',name:r.name,x:[\'Teaching\',\'Research\',\'Citations\',\'Intl outlook\',\'Industry\'],y:metrics.map(m=>finite(r[\'score_\'+m+\'_lower\'])?r[\'score_\'+m+\'_lower\']:null),hovertemplate:esc(r.name)+\'<br>%{x}: %{y}<extra></extra>\'}});plot(\'compare\',traces,{barmode:\'group\',yaxis:{...layout.yaxis,title:\'Lower score bound\',range:[0,100]},legend:{orientation:\'h\',y:1.18},margin:{l:50,r:10,t:55,b:40},annotations:traces.length?[]:[{text:\'Choose institutions above\',showarrow:false,x:0.5,y:0.5,xref:\'paper\',yref:\'paper\'}]})}\nfunction switchView(view){currentView=view;$(\'globe\').style.display=view===\'globe\'?\'block\':\'none\';$(\'flat\').style.display=view===\'flat\'?\'block\':\'none\';for(const [id,v]of[[\'globeBtn\',\'globe\'],[\'flatBtn\',\'flat\']]){$(id).classList.toggle(\'active\',view===v);$(id).setAttribute(\'aria-pressed\',String(view===v))}if(view===\'flat\'&&window.Plotly)Plotly.Plots.resize(\'flat\')}\nfunction csvValue(v){let s=String(v??\'\');if(/^[=+@-]/.test(s))s="\'"+s;return \'"\'+s.replace(/"/g,\'""\')+\'"\'}\n$(\'export\').onclick=()=>{const cols=[\'name\',\'location\',\'rank_original\',\'academic_status\',\'score_overall_lower\',\'score_overall_upper\'];const csv=[cols.join(\',\'),...selected.map(r=>cols.map(c=>csvValue(r[c])).join(\',\'))].join(\'\\r\\n\');const url=URL.createObjectURL(new Blob([\'\\ufeff\'+csv],{type:\'text/csv;charset=utf-8\'}));const a=document.createElement(\'a\');a.href=url;a.download=\'universityscope_selection.csv\';a.click();setTimeout(()=>URL.revokeObjectURL(url),1000)};\nfor(const name of [...new Set(data.map(r=>r.location))].sort()){const o=document.createElement(\'option\');o.value=name;o.textContent=name;$(\'country\').appendChild(o)}\nconst options=data.map((r,i)=>({r,i})).sort((a,b)=>a.r.name.localeCompare(b.r.name));for(let slot=0;slot<4;slot++){const label=document.createElement(\'label\');label.textContent=\'Institution \'+(slot+1);const select=document.createElement(\'select\');select.setAttribute(\'aria-label\',\'Comparison institution \'+(slot+1));select.innerHTML=\'<option value="">Choose institution</option>\'+options.map(({r,i})=>`<option value="${i}">${esc(r.name)} · ${esc(r.location)}</option>`).join(\'\');select.style.maxWidth=\'280px\';select.onchange=()=>{comparison[slot]=select.value;compare()};label.appendChild(select);$(\'compareControls\').appendChild(label)}\nfor(const id of [\'search\',\'country\',\'cohort\',\'metric\'])$(id).addEventListener(id===\'search\'?\'input\':\'change\',render);$(\'reset\').onclick=()=>{$(\'search\').value=\'\';$(\'country\').value=\'\';$(\'cohort\').value=\'\';render()};$(\'globeBtn\').onclick=()=>switchView(\'globe\');$(\'flatBtn\').onclick=()=>switchView(\'flat\');$(\'source\').textContent=payload.provenance+\' · Country-level coverage, not all universities worldwide. Source scores are screening evidence, not partnership readiness. Charts use lower score bounds; zero and missing remain distinct.\';\nasync function start(){if(!window.Plotly){$(\'status\').textContent=\'Chart library could not load. Check network access to jsDelivr and rerun this cell.\';return}render();try{if(!window.Globe||!window.topojson)throw Error(\'Globe libraries unavailable\');const response=await fetch(\'https://cdn.jsdelivr.net/npm/world-atlas@2.0.2/countries-110m.json\');if(!response.ok)throw Error(\'World boundaries unavailable\');const world=await response.json();features=topojson.feature(world,world.objects.countries).features;globe=Globe()($(\'globe\')).backgroundColor(\'#080d1a\').showAtmosphere(true).atmosphereColor(\'#67d6dd\').atmosphereAltitude(0.15).polygonSideColor(()=>\'#16424e\').polygonStrokeColor(()=>\'#44617c\').polygonsTransitionDuration(450);globe.width($(\'map\').clientWidth).height($(\'map\').clientHeight);globe.pointOfView({lat:22,lng:15,altitude:2.25});globe.controls().autoRotate=!matchMedia(\'(prefers-reduced-motion: reduce)\').matches;globe.controls().autoRotateSpeed=0.25;globe.controls().enableZoom=true;updateMap();new ResizeObserver(()=>globe.width($(\'map\').clientWidth).height($(\'map\').clientHeight)).observe($(\'map\'));}catch(error){$(\'status\').textContent=\'3D globe unavailable: \'+error.message+\'. The 2D map and data explorer remain available.\';switchView(\'flat\')}}\nstart();\n</script></body></html>\n', encoding='utf-8')
"""Generate a standalone dashboard and embed it directly in Colab/Jupyter."""
from pathlib import Path
import json
import html
import pandas as pd

def build_dashboard(df, provenance, template='dashboard.html', output='universityscope_dashboard.html'):
    import pycountry
    aliases = {'United States': 'USA', 'United Kingdom': 'GBR', 'South Korea': 'KOR', 'Russia': 'RUS', 'Taiwan': 'TWN', 'Hong Kong': 'HKG', 'Iran': 'IRN', 'Vietnam': 'VNM', 'Türkiye': 'TUR', 'Kosovo': None}
    countries = {}
    for name in df['location'].unique():
        try:
            country = pycountry.countries.get(alpha_3=aliases[name]) if name in aliases else pycountry.countries.lookup(name)
            countries[name] = {'iso': country.alpha_3, 'numeric': country.numeric} if country else None
        except LookupError:
            countries[name] = None
    columns = ['name', 'location', 'rank_original', 'rank_lower_bound', 'academic_status']
    columns += [c for c in df if c.startswith('score_') and (c.endswith('_lower') or c.endswith('_upper'))]
    data = json.loads(df[columns].to_json(orient='records'))
    payload = json.dumps({'rows': data, 'countries': countries, 'provenance': provenance}, ensure_ascii=False, allow_nan=False).replace('<', '\\u003c')
    document = Path(template).read_text().replace('__PAYLOAD__', payload)
    Path(output).write_text(document)
    return document

def show_dashboard(document, height=1050):
    from IPython.display import HTML, display
    display(HTML('<iframe title="UniversityScope interactive dashboard" style="width:100%;height:' + str(height) + 'px;border:0;border-radius:16px" sandbox="allow-scripts allow-same-origin allow-downloads" srcdoc="' + html.escape(document, quote=True) + '"></iframe>'))

document = build_dashboard(df, provenance)
show_dashboard(document, height=1150)
print('Dashboard displayed above. Standalone output: universityscope_dashboard.html')
